# 11 — Итоговая сводка результатов (Глава 3 ВКР)

Финальные таблицы для разделов 3.2, 3.3, 3.5, 3.7 ВКР. Все значения берём из `reports/test_evaluation.json` (test set, 972 v4.0-записи) и из метрик последней эпохи Stage 2 (записаны вручную в ноутбуке `04_evaluation.ipynb`).

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

with open(ROOT / 'reports' / 'test_evaluation.json', 'r', encoding='utf-8') as fh:
    data = json.load(fh)
agg = data['aggregated']
per_metric = data['per_metric']

## Раздел 1. Итоговые метрики качества (для раздела 3.2 ВКР)

In [2]:
summary = pd.DataFrame([
    ('Macro-F1 (12 метрик)',          f"{agg['macro_f1']:.4f}"),
    ('Vector Accuracy (11 метрик)',   f"{agg['vector_accuracy']:.4f}"),
    ('Среднее число правильных метрик',
         f"{agg['metrics_correct_avg']:.2f} / 11 ({agg['metrics_correct_avg']/11:.1%})"),
    ('MAE по CVSS-баллу',            f"{agg['score_mae']:.2f} ({agg['score_mae']/10:.1%} шкалы)"),
    ('RMSE по CVSS-баллу',           f"{agg['score_rmse']:.2f}"),
    ('Severity Accuracy',            f"{agg['severity_accuracy']:.4f}"),
    ('Severity Within ±1',           f"{agg['severity_within_one']:.4f}"),
    ('Размер test set',              f"{agg['samples_evaluated']} CVSS v4.0 записей"),
], columns=['Показатель', 'Значение'])
(ROOT / 'reports' / 'final_summary.md').write_text(
    '# Итоговые метрики качества (test set)\n\n' +
    summary.to_markdown(index=False) + '\n',
    encoding='utf-8',
)
summary

,Показатель,Значение
0,Macro-F1 (12 метрик),0.7090
1,Vector Accuracy (11 метрик),0.3992
2,Среднее число правильных метрик,9.39 / 11 (85.4%)
3,MAE по CVSS-баллу,1.17 (11.7% шкалы)
4,RMSE по CVSS-баллу,1.98
5,Severity Accuracy,0.6739
6,Severity Within ±1,0.9208
7,Размер test set,972 CVSS v4.0 записей


## Раздел 2. Per-metric F1 / Accuracy (для раздела 3.3 ВКР)

In [3]:
FULL_NAMES = {
    'AV': 'Attack Vector',
    'AC': 'Attack Complexity',
    'AT': 'Attack Requirements',
    'PR': 'Privileges Required',
    'UI': 'User Interaction',
    'VC': 'Vulnerable System Confidentiality',
    'VI': 'Vulnerable System Integrity',
    'VA': 'Vulnerable System Availability',
    'SC': 'Subsequent System Confidentiality',
    'SI': 'Subsequent System Integrity',
    'SA': 'Subsequent System Availability',
    'E':  'Exploit Maturity',
}
per_metric_df = pd.DataFrame([
    {
        'Метрика': m,
        'Полное название': FULL_NAMES[m],
        'F1 (macro)': round(per_metric[m]['f1_macro'], 4),
        'Accuracy': round(per_metric[m]['accuracy'], 4),
        'Support': per_metric[m]['support'],
    }
    for m in FULL_NAMES
])
(ROOT / 'reports' / 'per_metric_summary.md').write_text(
    '# Per-metric F1 / Accuracy на test set\n\n' +
    per_metric_df.to_markdown(index=False) + '\n',
    encoding='utf-8',
)
per_metric_df

,Метрика,Полное название,F1 (macro),Accuracy,Support
0,AV,Attack Vector,0.5175,0.9023,972
1,AC,Attack Complexity,0.7482,0.9300,972
2,AT,Attack Requirements,0.7433,0.8837,972
3,PR,Privileges Required,0.6308,0.7212,972
4,UI,User Interaction,0.6414,0.8827,972
5,VC,Vulnerable System Confidentiality,0.7886,0.7942,972
6,VI,Vulnerable System Integrity,0.8198,0.8241,972
7,VA,Vulnerable System Availability,0.7946,0.7994,972
8,SC,Subsequent System Confidentiality,0.6228,0.8704,972
9,SI,Subsequent System Integrity,0.6705,0.8909,972


## Раздел 3. Стабильность модели — val vs test (для раздела 3.5 ВКР)

In [4]:
# F1 на val на последней эпохе Stage 2 (взято из логов обучения,
# дублирует значения из notebooks/04_evaluation.ipynb)
val_metrics_stage2 = {
    'AV': 0.534, 'AC': 0.724, 'AT': 0.744, 'PR': 0.629,
    'UI': 0.629, 'VC': 0.805, 'VI': 0.813, 'VA': 0.823,
    'SC': 0.648, 'SI': 0.635, 'SA': 0.660, 'E':  0.881,
}
stability = pd.DataFrame([
    {
        'Метрика': m,
        'F1 val (epoch 18)': round(val_metrics_stage2[m], 3),
        'F1 test': round(per_metric[m]['f1_macro'], 3),
        'Δ': round(per_metric[m]['f1_macro'] - val_metrics_stage2[m], 3),
    }
    for m in val_metrics_stage2
])
(ROOT / 'reports' / 'stability_val_vs_test.md').write_text(
    '# Стабильность модели: val vs test\n\n' +
    stability.to_markdown(index=False) + '\n',
    encoding='utf-8',
)
stability

,Метрика,F1 val (epoch 18),F1 test,Δ
0,AV,0.534,0.517,-0.017
1,AC,0.724,0.748,0.024
2,AT,0.744,0.743,-0.001
3,PR,0.629,0.631,0.002
4,UI,0.629,0.641,0.012
5,VC,0.805,0.789,-0.016
6,VI,0.813,0.820,0.007
7,VA,0.823,0.795,-0.028
8,SC,0.648,0.623,-0.025
9,SI,0.635,0.670,0.035


## Раздел 4. Финальная Markdown-сводка

In [5]:
final_md = '\n\n'.join([
    '# Итоговые результаты экспериментов\n',
    '## Интегральные метрики качества\n',
    summary.to_markdown(index=False),
    '## Per-metric качество (test set)\n',
    per_metric_df.to_markdown(index=False),
    '## Стабильность: val vs test\n',
    stability.to_markdown(index=False),
])
(ROOT / 'reports' / 'final_results.md').write_text(final_md, encoding='utf-8')
print('Сохранено в reports/final_results.md')

Сохранено в reports/final_results.md


## Раздел 5. Показательные примеры для приложения к ВКР

Берём `reports/error_analysis/test_predictions.parquet` (создан в ноутбуке 10) и выбираем:
- 2 примера с идеальным предсказанием (11/11 метрик),
- 2 примера с частичными ошибками (8–9 / 11),
- 1 пример с сильной ошибкой (< 6 / 11).

Для каждого даётся true_vector, pred_vector, средняя уверенность и разбор низкоконфидентных метрик.

In [6]:
PRED_PATH = ROOT / 'reports' / 'error_analysis' / 'test_predictions.parquet'
if not PRED_PATH.exists():
    raise FileNotFoundError(
        f'Сначала прогоните notebook 10_error_analysis.ipynb — '
        f'нет файла {PRED_PATH}'
    )
preds = pd.read_parquet(PRED_PATH)
print('Загружено предсказаний:', len(preds))
preds[['cve_id', 'cwe_id', 'num_metrics_correct', 'true_severity',
       'pred_severity', 'confidence_avg']].head()

Загружено предсказаний: 972


,cve_id,cwe_id,num_metrics_correct,true_severity,pred_severity,confidence_avg
0,CVE-2024-52324,CWE-242,10,Critical,Critical,0.962724
1,CVE-2024-28050,CWE-284,10,Medium,Medium,0.877806
2,CVE-2024-9277,CWE-1333,8,Medium,Medium,0.883315
3,CVE-2024-11303,CWE-22,8,High,Critical,0.875400
4,CVE-2025-32827,CWE-89,11,High,High,0.926162


In [7]:
from src.data_preparation.cvss_vector_parser import V4_METRIC_ORDER
BASE = [m for m in V4_METRIC_ORDER if m != 'E']

def pick(df, n_correct_eq=None, n_correct_lt=None, n_correct_ge=None, n=1, seed=42):
    sub = df.copy()
    if n_correct_eq is not None:
        sub = sub[sub.num_metrics_correct == n_correct_eq]
    if n_correct_lt is not None:
        sub = sub[sub.num_metrics_correct < n_correct_lt]
    if n_correct_ge is not None:
        sub = sub[sub.num_metrics_correct >= n_correct_ge]
    sub = sub[sub.description.str.len() > 80]
    return sub.sample(n=min(n, len(sub)), random_state=seed)

perfect = pick(preds, n_correct_eq=11, n=2, seed=42)
partial = pick(preds[(preds.num_metrics_correct >= 8) & (preds.num_metrics_correct <= 9)], n=2, seed=123)
bad = pick(preds, n_correct_lt=6, n=1, seed=7)
showcase = pd.concat([perfect, partial, bad]).reset_index(drop=True)
showcase[['cve_id', 'cwe_id', 'num_metrics_correct',
          'true_severity', 'pred_severity', 'true_score', 'pred_score']]

C:\Users\Артём\Desktop\diplom\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,cve_id,cwe_id,num_metrics_correct,true_severity,pred_severity,true_score,pred_score
0,CVE-2024-6115,CWE-434,11,Medium,Medium,6.9,6.9
1,CVE-2024-12928,CWE-74,11,Medium,Medium,5.3,5.3
2,CVE-2025-9235,CWE-79,9,Low,Medium,2.0,5.5
3,CVE-2026-21989,CWE-20,8,High,None,8.3,0.0
4,CVE-2024-52276,CWE-451,5,High,High,8.2,7.1


In [8]:
lines = ['# Показательные примеры предсказаний (приложение к ВКР)\n']
category_names = [
    'Идеальное предсказание (11/11 метрик)',
    'Идеальное предсказание (11/11 метрик)',
    'Частичная ошибка (8–9 / 11 метрик)',
    'Частичная ошибка (8–9 / 11 метрик)',
    'Грубая ошибка (< 6 / 11 метрик)',
]
for idx, row in showcase.iterrows():
    cat = category_names[idx] if idx < len(category_names) else 'Прочее'
    lines.append(f'## Пример {idx+1}. {cat}\n')
    lines.append(f'- **CVE:** {row.cve_id}')
    lines.append(f'- **CWE:** {row.cwe_id}')
    lines.append(f'- **Severity:** true = {row.true_severity}, pred = {row.pred_severity}')
    lines.append(f'- **Score:** true = {row.true_score:.1f}, pred = {row.pred_score:.1f}')
    lines.append(f'- **Метрик правильно:** {int(row.num_metrics_correct)} / 11')
    lines.append(f'- **Средняя уверенность:** {row.confidence_avg:.3f}')
    lines.append('')
    lines.append('**Описание:**')
    lines.append('')
    desc = str(row.description).strip()
    lines.append(f'> {desc[:600]}{"…" if len(desc) > 600 else ""}')
    lines.append('')
    lines.append(f'**True vector:** `{row.true_vector}`  \n**Pred vector:** `{row.pred_vector}`')
    lines.append('')
    wrong = []
    for m in BASE:
        if row[f'true_{m}'] != row[f'pred_{m}']:
            wrong.append(
                f'  - **{m}:** true=`{row[f"true_{m}"]}`, '
                f'pred=`{row[f"pred_{m}"]}` (confidence={row[f"conf_{m}"]:.3f})'
            )
    if wrong:
        lines.append('**Ошибки модели:**')
        lines.extend(wrong)
    else:
        lines.append('**Ошибок нет** — модель попала во все 11 базовых метрик.')
    lines.append('')
    low = [m for m in V4_METRIC_ORDER if row[f'conf_{m}'] < 0.7]
    if low:
        lines.append(f'**Низкая уверенность** (<0.7): {", ".join(low)}')
    lines.append('')
    lines.append('---\n')

showcase_md = '\n'.join(lines)
(ROOT / 'reports' / 'showcase_examples.md').write_text(showcase_md, encoding='utf-8')
print('Сохранено в reports/showcase_examples.md ({} символов)'.format(len(showcase_md)))

Сохранено в reports/showcase_examples.md (5048 символов)


In [9]:
print('Создано:')
for f in ['final_summary.md', 'per_metric_summary.md',
         'stability_val_vs_test.md', 'final_results.md',
         'showcase_examples.md']:
    p = ROOT / 'reports' / f
    print(f' • {p.relative_to(ROOT)} ({p.stat().st_size} байт)')

Создано:
 • reports\final_summary.md (766 байт)
 • reports\per_metric_summary.md (1339 байт)
 • reports\stability_val_vs_test.md (876 байт)
 • reports\final_results.md (3048 байт)
 • reports\showcase_examples.md (6195 байт)
